# Error analysis of the ensemble's false positives

Draws the reproducible sample of ensemble errors used for the failure-mode annotation, joins the manual labels shipped in `DATA/outputs/fp_error_annotation.csv`, and rebuilds the failure-mode table of the paper's quantitative error analysis appendix.

In [1]:
import hashlib

import pandas as pd

bench = pd.read_csv("../../DATA/outputs/benchmark.csv")
oof = pd.read_csv("../../DATA/outputs/predictions/supervised_oof/ensemble_oof.csv")
for df in (bench, oof):
    df["pred_art"] = df["pred_art"].astype(str)
m = bench.merge(oof, on=["decision_id", "chunk_id", "pred_art"])

# 238 ensemble errors; random but reproducible sample: sort the pair ids by their
# md5 fingerprint (uniform, content-agnostic order) and keep the first 100 errors.
# The false positives among them (n=25) were annotated with a failure mode.
err = m[m.prediction != m.gold].copy()
err["pair_id"] = err.decision_id + "|" + err.chunk_id.astype(str) + "|" + err.pred_art
err["key"] = err.pair_id.map(lambda s: hashlib.md5(s.encode()).hexdigest())
sample = err.sort_values("key").head(100)
len(err), len(sample)

(238, 100)

In [2]:
labels = pd.read_csv("../../DATA/outputs/fp_error_annotation.csv")
labels["pred_art"] = labels["pred_art"].astype(str)
fp = sample[(sample.prediction == 1) & (sample.gold == 0)]
fp = fp.merge(labels, on=["decision_id", "chunk_id", "pred_art"], validate="one_to_one")
assert len(fp) == 25

tab = fp.groupby("failure_mode").agg(
    N=("gold", "size"),
    disagreed=("agreement", lambda a: int((a == 0).sum())),
    conf_mean=("proba", "mean"),
    conf_median=("proba", "median"),
)
tab["pct"] = 100 * tab.N / tab.N.sum()
tab.loc["total"] = [tab.N.sum(), tab.disagreed.sum(), fp.proba.mean(), fp.proba.median(), 100]
tab.round(2)

,N,disagreed,conf_mean,conf_median,pct
failure_mode,,,,,
other,1.0,0.0,0.71,0.71,4.0
statutory_language_not_applied,14.0,8.0,0.75,0.77,56.0
wrong_rule,10.0,6.0,0.74,0.76,40.0
total,25.0,14.0,0.74,0.76,100.0
